# HRAI API demo demo notebook


In [ ]:
!pip install --upgrade pip

In [ ]:
!pip install -r requirements.txt

In [1]:
import os, json, random, textwrap, requests
from pathlib import Path

from extraction import text_to_ngrams
from config import config

from dotenv import load_dotenv
import rich

load_dotenv()

True

In [ ]:
# pro rychlejší stáhnutí encoderu (za předpokladu že není použitý git lfs) doporučuji nastavit HF_TOKEN:
os.environ['HF_TOKEN'] = ''

## definice

In [2]:
resume_path = Path(os.path.join(config.data_dir, 'cs_resumes.json')) # součást původního datasetu, nepoužito při trénování nikterého z modelů

def pick_resume():
    resumes = json.loads(resume_path.read_text(encoding='utf-8'))
    resume = random.choice(resumes)
    return resume.strip()


def call_resume_endpoint(payload: dict) -> dict:
    r = requests.post('http://0.0.0.0:8000/resume', json=payload)
    r.raise_for_status()
    return r.json()

def call_post_entities(entities: list[dict], top_n: int = 5, extra_skill_limit: int = 10):
    payload = {
        'entities': entities,
        'top_n': top_n,
        'extra_skill_limit': extra_skill_limit,
    }
    resp = requests.post('http://0.0.0.0:8000/post', json=payload)
    resp.raise_for_status()
    return resp.json()

def print_lookup_data(data: dict, max_skills: int = 8, max_domains: int = 3):
    skills = data.get('Schopnosti a dovednosti:', [])
    print(f"(zobrazeno {max_skills} z nalezených {len(skills)}):")
    for skill in skills[:max_skills]:
        print(f"  - {skill['label']} ( skóre podobnosti: {skill['score']:.3f}, zdroj: {skill['source']}, text: {skill['source_text']})")
    domains = data.get('domain_recommendations', [])
    print()
    print(f"nejlépe skórující povolání z různých oborů ESCO: (top {min(max_domains, len(domains))}/{len(domains)}):")
    for dom in domains[:max_domains]:
        occ = dom['occupation']
        print(f"  - ID oboru {dom['domain_code']} | {occ['label']} (ISCO {occ['isco_code']}, skóre (ranking) {occ['score']:.3f})")
        for extra in dom.get('extra_skills', [])[:5]:
            print(f"(možná) chybějící dovednost: \n    {extra['label']}    důležitost: ({extra['relation_type']}\n skóre (ranking): {extra['score']:.3f})")
    if data.get('occupation_suggestions'):
        print('\nESCO kategorie pro specifikovanou pozici:')
        for s in data['occupation_suggestions']:
            occ = s['occupation']
            print(f"  - {occ['label']} (skóre podobnosti: {occ['score']:.3f})")


## Testování

In [3]:
text = pick_resume()
print(textwrap.shorten(text, width=600, placeholder='...'))

Hlavní body Osmiletý vojenský veterán se sedmiletou praxí v oblasti vzdělávání. Čtyři roky zkušeností s výukou umění v celé farnosti Caddo. Zkušenosti s výukou umění zahrnují službu nadaným studentům umění zapsaným do programu Talented Arts Program (TAP) na školách Caddo. Působil také jako učitel výtvarné výchovy na základní škole Forest Hill a základní škole Judson. Působím také jako učitelka výtvarné výchovy v mimoškolním programu Volunteers of America na základní škole Forest Hill. Kreslím a maluji již od základní školy. Prodávám také svá soukromá umělecká díla jako umělkyně na volné...


In [4]:
ngrams = text_to_ngrams(text)
for phrase in ngrams:
    print(phrase)

výukou umění
nákladních vozidel zprostředkovateli
nákupem
zdrojů
art
spolumajitel manažer logistiky
položky pro nejlepší
realizovatelných logistických strategií
prodej a silné
springs co zaměření
logistických operací
zdrojů -12 27
větší
identifikoval požadavky
plány
realizovatelných
údržba operace
absolvoval únor 09
organizovaný spolehlivý
umění v celé
programu volunteers of
springs co koncentrace
veřejných
zapsaným do programu
e maily
poskytoval zákaznický servis
povolenky
snadno srozumitelný
000 usd ročně
nákladů díky
art program
statusem nákladní
potřeby
speciálními
schválení nákupu mimo
věd business
srozumitelný
plán
služeb září
službu nadaným studentům
položky
dětmi v různých
odpovídá za efektivní
farních školách
zaměstnanců
recenzí literatury rozhovorů
program vzdělávací koordinátor
střední školy
maily dodavatelům
e maily dodavatelům
různých farních
monitoroval rozpočty zákaznických
logistiky zásobování
bakalář věd business
režijních
vydávání disciplinárních
stát plat 2
umělecký 

In [ ]:
from query import extract_from_resume

results = extract_from_resume(text) # docela to trvá, hlavně na cpu

In [10]:
for r in results:
    print(
        f'{r.label}\n'
        f'skore: {r.cosine_score}\n'
        f'typ: {r.entity_type}\n'
    )

recepční
skore: 1.0
typ: occupation

řešit problémy
skore: 0.9999999403953552
typ: skill

účetní
skore: 0.9590039849281311
typ: occupation

nákupčí
skore: 0.9291298389434814
typ: occupation

koordinátor vzdělávacích programů/koordinátorka vzdělávacích programů
skore: 0.9134435653686523
typ: occupation

ředitel základní školy/ředitelka základní školy
skore: 0.9041699171066284
typ: occupation

projektový manažer/projektová manažerka
skore: 0.9017610549926758
typ: occupation

manažer logistiky a distribuce/manažerka logistiky a distribuce
skore: 0.896246075630188
typ: occupation

projektový manažer/projektová manažerka
skore: 0.8848555088043213
typ: occupation

učitel výtvarných umění/učitelka výtvarných umění
skore: 0.8847607970237732
typ: occupation

řidič nákladního automobilu/řidička nákladního automobilu
skore: 0.8846086859703064
typ: occupation

marketingový manažer/marketingová manažerka
skore: 0.8318983912467957
typ: occupation

pracovník služeb pro zákazníky/pracovnice služeb pro

### hledání N-gramů z CV přímo v indexu
- parsování modelem Czech-pdt or ÚFAL

#### pozice + CV (ngram pipeline)
-> ESCO pozice nejbližší ke vztupu + doporučené dovednosti které nejsou blízké těm, které v CV jsou

In [ ]:
job_title = input('Sem může uživatel napsat třeba svojí "práci snů": ')
payload = {
    'resume_text': text,
    'job_title': job_title,
    'top_n': 1,
}
resp_data = call_resume_endpoint(payload)
rich.print(resp_data)

#### CV input
-> Pokusí se odhadnout oblast uživatele (např. životopis na brigádu v kuchyni), a na základě toho doporučit relevantní dovednosti

In [ ]:
payload = {
    'resume_text': text,
    'job_title': None,
    'top_n': 1,
}
resp_data = call_resume_endpoint(payload)
rich.print(resp_data)

### Manuální vstup

In [ ]:
manual_skills = input('vlož seznam dovedností')  # replace with any skill list


## Dotazy

In [ ]:
def query_single_entity(text: str, entity_type: str, top_n: int = 5):
    entities = [{'text': text, 'entity_type': entity_type}]
    results = call_post_entities(entities, top_n=top_n)
    print(f"Top matches in '{entity_type}' index for '{text}':")
    for match in results:
        occ = match['occupation']
        print(f"  - {occ['label']} (ISCO {occ['isco_code']}, score {occ['score']:.3f})")
    return results

### Práce

In [ ]:
q = input()
query_single_entity(q, entity_type='occupation', top_n=5)

### Schopnost/Dovednost

In [ ]:
q = input()
query_single_entity(q, entity_type='skill', top_n=5)

### Prace - obecnější kategorie

In [ ]:
q = input()
query_single_entity(q, entity_type='isco_group', top_n=5)

### Schopnost/Dovednost - obecnější kategorie

In [ ]:
q = input()
query_single_entity(q, entity_type='skill_group', top_n=5)